# Lab: Custom Training Loops in Keras — worked study copy

Answers prepared for revision with AI assistance. IBM / Skills Network source notices and embedded instructor explanations are retained. This is not a course submission.

Code completion and execution are tracked separately in the [solution progress report](../../../docs/SOLUTION_PROGRESS.md). Training may require a separate environment, external data, compute or provider credentials.

[Original reference](../../../courses/03-keras-and-tensorflow/notebooks/lab-custom-training-loops-in-keras.ipynb) · [Course notes](../../../courses/03-keras-and-tensorflow/README.md)

**BOUNDED SMOKE RUN:** seed 42; CSVs sampled to at most 5,000 rows; one grid setting with two folds; training budgets capped; DataLoaders limited to 12 batches. These outputs are execution checks, not full-run model evaluations. Bundled TensorFlow datasets use 512 training / 256 test examples; fit inputs use at most 512 examples or eight dataset batches.

In [1]:
import os, itertools, random
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
import numpy as np
np.random.seed(42)
random.seed(42)
import pandas as pd
_read_csv = pd.read_csv
def _bounded_csv(*args, **kwargs):
    data = _read_csv(*args, **kwargs)
    return data.sample(n=5000,random_state=42) if len(data)>5000 else data
pd.read_csv = _bounded_csv
from sklearn.model_selection import GridSearchCV
_grid_fit = GridSearchCV.fit
def _bounded_grid(self, *args, **kwargs):
    grids = self.param_grid if isinstance(self.param_grid,list) else [self.param_grid]
    self.param_grid = [{k:list(v)[:1] for k,v in grid.items()} for grid in grids[:1]]
    self.cv = 2
    self.n_jobs = 1
    self.verbose = 0
    return _grid_fit(self,*args,**kwargs)
GridSearchCV.fit = _bounded_grid
try:
    import torch
except ImportError:
    torch = None
if torch is not None:
    torch.manual_seed(42)
    torch.set_num_threads(1)
    _loader_iter = torch.utils.data.DataLoader.__iter__
    def _bounded_loader(self):
        return itertools.islice(_loader_iter(self),12)
    torch.utils.data.DataLoader.__iter__ = _bounded_loader
import tensorflow as tf
tf.random.set_seed(42)
for _dataset in [tf.keras.datasets.mnist,tf.keras.datasets.fashion_mnist,tf.keras.datasets.cifar10]:
    _original_load = _dataset.load_data
    def _small_load(*args,_load=_original_load,**kwargs):
        (x,y),(xt,yt) = _load(*args,**kwargs)
        return (x[:512],y[:512]),(xt[:256],yt[:256])
    _dataset.load_data = _small_load
_keras_fit = tf.keras.Model.fit
def _small_fit(self,*args,**kwargs):
    args = list(args)
    def _limit(value):
        if isinstance(value,(np.ndarray,pd.DataFrame,pd.Series)):return value[:512]
        if isinstance(value,(tuple,list)) and value and all(isinstance(x,np.ndarray) for x in value):return type(value)(x[:512] for x in value)
        if isinstance(value,tf.data.Dataset):return value.take(8)
        return value
    for i in range(min(2,len(args))):args[i]=_limit(args[i])
    for key in ['x','y','validation_data']:
        if key in kwargs:kwargs[key]=_limit(kwargs[key])
    kwargs['epochs']=min(2,kwargs.get('epochs',2))
    data=args[0] if args else kwargs.get('x')
    if hasattr(data,'class_indices'):
        kwargs['steps_per_epoch']=min(2,len(data))
        if hasattr(kwargs.get('validation_data'),'class_indices'):
            kwargs['validation_steps']=min(2,len(kwargs['validation_data']))
    return _keras_fit(self,*args,**kwargs)
tf.keras.Model.fit = _small_fit


Matplotlib is building the font cache; this may take a moment.


<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Custom Training Loops in Keras**


Estimated time needed: **30** minutes


In this lab, you will learn to implement a basic custom training loop in Keras. 


## Objectives

By the end of this lab, you will: 

- Set up the environment 

- Define the neural network model 

- Define the Loss Function and Optimizer 

- Implement the custom training loop 

- Enhance the custom training loop by adding an accuracy metric to monitor model performance 

- Implement a custom callback to log additional metrics and information during training


----


## Step-by-Step Instructions:


### Exercise 1: Basic custom training loop: 

#### 1. Set Up the Environment:

- Import necessary libraries. 

- Load and preprocess the MNIST dataset. 


In [2]:
import os
import warnings
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = (x_train / 255.0, x_test / 255.0)
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

#### 2. Define the model: 

Create a simple neural network model with a Flatten layer followed by two Dense layers. 


In [3]:
model = Sequential([Flatten(input_shape=(28, 28)), Dense(128, activation='relu'), Dense(10)])

#### 3. Define Loss Function and Optimizer: 

- Use Sparse Categorical Crossentropy for the loss function. 
- Use the Adam optimizer. 


In [4]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

#### 4. Implement the Custom Training Loop: 

- Iterate over the dataset for a specified number of epochs. 
- Compute the loss and apply gradients to update the model's weights. 


In [5]:
epochs = 2
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
for epoch in itertools.islice(range(epochs), 2):
    print(f'Start of epoch {epoch + 1}')
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)
            loss_value = loss_fn(y_batch_train, logits)
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')

Start of epoch 1
Epoch 1 Step 0: Loss = 2.320596694946289
Start of epoch 2
Epoch 2 Step 0: Loss = 1.202382206916809


### Exercise 2: Adding Accuracy Metric:

Enhance the custom training loop by adding an accuracy metric to monitor model performance. 

#### 1. Set Up the Environment: 

Follow the setup from Exercise 1. 


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = (x_train / 255.0, x_test / 255.0)
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

#### 2. Define the Model: 
Use the same model as in Exercise 1. 


In [7]:
model = Sequential([Flatten(input_shape=(28, 28)), Dense(128, activation='relu'), Dense(10)])

#### 3. Define the loss function, optimizer, and metric: 

- Use Sparse Categorical Crossentropy for the loss function and Adam optimizer. 

- Add Sparse Categorical Accuracy as a metric. 


In [8]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

#### 4. Implement the custom training loop with accuracy: 

Track the accuracy during training and print it at regular intervals. 


In [9]:
epochs = 2
for epoch in itertools.islice(range(epochs), 2):
    print(f'Start of epoch {epoch + 1}')
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)
            loss_value = loss_fn(y_batch_train, logits)
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        accuracy_metric.update_state(y_batch_train, logits)
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    accuracy_metric.reset_state()

Start of epoch 1
Epoch 1 Step 0: Loss = 2.3598861694335938 Accuracy = 0.03125
Start of epoch 2
Epoch 2 Step 0: Loss = 1.086883783340454 Accuracy = 0.8125


### Exercise 3: Custom Callback for Advanced Logging: 

Implement a custom callback to log additional metrics and information during training. 

#### 1. Set Up the Environment: 

Follow the setup from Exercise 1.


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = (x_train / 255.0, x_test / 255.0)
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

#### 2. Define the Model: 

Use the same model as in Exercise 1. 


In [11]:
model = Sequential([Flatten(input_shape=(28, 28)), Dense(128, activation='relu'), Dense(10)])

#### 3. Define Loss Function, Optimizer, and Metric: 

- Use Sparse Categorical Crossentropy for the loss function and Adam optimizer. 

- Add Sparse Categorical Accuracy as a metric. 


In [12]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

#### 4. Implement the custom training loop with custom callback: 

Create a custom callback to log additional metrics at the end of each epoch.


In [13]:
from tensorflow.keras.callbacks import Callback

class CustomCallback(Callback):

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f"End of epoch {epoch + 1}, loss: {logs.get('loss')}, accuracy: {logs.get('accuracy')}")

In [14]:
epochs = 2
custom_callback = CustomCallback()
for epoch in itertools.islice(range(epochs), 2):
    print(f'Start of epoch {epoch + 1}')
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)
            loss_value = loss_fn(y_batch_train, logits)
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        accuracy_metric.update_state(y_batch_train, logits)
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy': accuracy_metric.result().numpy()})
    accuracy_metric.reset_state()

Start of epoch 1
Epoch 1 Step 0: Loss = 2.3059215545654297 Accuracy = 0.1875
End of epoch 1, loss: 1.6603089570999146, accuracy: 0.505859375
Start of epoch 2
Epoch 2 Step 0: Loss = 1.061358094215393 Accuracy = 0.875


End of epoch 2, loss: 1.0924923419952393, accuracy: 0.81640625


### Exercise 4: Add Hidden Layers 

Next, you will add a couple of hidden layers to your model. Hidden layers help the model learn complex patterns in the data. 


In [15]:
from tensorflow.keras.layers import Input, Dense
input_layer = Input(shape=(28, 28))
flatten = Flatten()(input_layer)
hidden_layer1 = Dense(64, activation='relu')(flatten)
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)

In the above code: 

`Flatten()` converts each 28×28 image into a 784-element 1D vector so it can be fed into Dense layers. 

`Dense(64, activation='relu')` creates a dense (fully connected) layer with 64 units and ReLU activation function. 

Each hidden layer takes the output of the previous layer as its input.


### Exercise 5: Define the output layer 

Finally, you will define the output layer. Suppose you are working on a binary classification problem, so the output layer will have one unit with a sigmoid activation function. 


In [16]:
output_layer = Dense(10, activation='softmax')(hidden_layer2)

In the above code: 

`Dense(10, activation='softmax')` creates the output layer with 10 neurons — one for each digit class (0–9). 

`softmax` activation ensures all 10 output values sum to 1, making them interpretable as class probabilities. 


### Exercise 6: Create the Model 

Now, you will create the model by specifying the input and output layers. 


In [17]:
model = Model(inputs=input_layer, outputs=output_layer)

In the above code: 

`Model(inputs=input_layer, outputs=output_layer)` creates a Keras model that connects the input layer to the output layer through the hidden layers. 


### Exercise 7: Compile the Model 

Before training the model, you need to compile it. You will specify the loss function, optimizer, and evaluation metrics. 


In [18]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In the above code: 

`optimizer='adam'` specifies the Adam optimizer, a popular choice for training neural networks. 

`loss='sparse_categorical_crossentropy'` is the correct loss for **multi-class classification** when labels are integers (0–9).

`metrics=['accuracy']` tells Keras to evaluate the model using accuracy during training. 


### Exercise 8: Train the Model 

You can now train the model on some training data. For this example, let's assume `X_train` is our training input data and `y_train` is the corresponding labels. 


In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import numpy as np
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = (x_train / 255.0, x_test / 255.0)
history = model.fit(x_train, y_train, epochs=2, batch_size=32)

Epoch 1/2


 1/16 ━━━━━━━━━━━━━━━━━━━━ 4s 288ms/step - accuracy: 0.1562 - loss: 2.3496

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2040 - loss: 2.2302  


Epoch 2/2


 1/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7188 - loss: 1.6800

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 955us/step - accuracy: 0.6881 - loss: 1.5612


In the above code: 

`X_train` and `y_train` are placeholders for your actual training data. 

`model.fit` trains the model for a specified number of epochs and batch size. 


### Exercise 9: Evaluate the Model 

After training, you can evaluate the model on test data to see how well it performs. 


In [20]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f'Test loss:     {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')

1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.7188 - loss: 1.2216

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6955 - loss: 1.2558 


Test loss:     1.2767
Test accuracy: 0.6758


In the above code: 

`model.evaluate` computes the loss and accuracy of the model on test data. 

`X_test` and `y_test` are placeholders for your actual test data. 


## Practice Exercises 

### Exercise 1: Basic Custom Training Loop 

#### Objective: Implement a basic custom training loop to train a simple neural network on the MNIST dataset. 

#### Instructions: 

- Set up the environment and load the dataset. 

- Define the model with a Flatten layer and two Dense layers. 

- Define the loss function and optimizer. 

- Implement a custom training loop to iterate over the dataset, compute the loss, and update the model's weights. 


In [21]:
(loop_x, loop_y), _ = tf.keras.datasets.mnist.load_data()
loop_x = loop_x.astype('float32') / 255
loop_data = tf.data.Dataset.from_tensor_slices((loop_x, loop_y)).shuffle(10000, seed=42).batch(64)

def make_loop_model():
    return tf.keras.Sequential([tf.keras.layers.Input((28, 28)), tf.keras.layers.Flatten(), tf.keras.layers.Dense(128, activation='relu'), tf.keras.layers.Dense(10)])
loop_model = make_loop_model()
loop_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
loop_optimizer = tf.keras.optimizers.Adam()
for epoch in itertools.islice(range(5), 2):
    epoch_loss = tf.keras.metrics.Mean()
    for batch_x, batch_y in loop_data:
        with tf.GradientTape() as tape:
            loss = loop_loss(batch_y, loop_model(batch_x, training=True))
        gradients = tape.gradient(loss, loop_model.trainable_variables)
        loop_optimizer.apply_gradients(zip(gradients, loop_model.trainable_variables))
        epoch_loss.update_state(loss)
    print(epoch + 1, float(epoch_loss.result()))

1 2.0576484203338623


2 1.3578609228134155


<details>
<summary>Click here for solution</summary> </br>

```python
# Import necessary libraries
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function and Optimizer
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 

# Step 4: Implement the Custom Training Loop
for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()}')


### Exercise 2: Adding Accuracy Metric 

#### Objective: Enhance the custom training loop by adding an accuracy metric to monitor model performance. 

#### Instructions: 

1. Set up the environment and define the model, loss function, and optimizer. 

2. Add Sparse Categorical Accuracy as a metric. 

3. Implement the custom training loop with accuracy tracking.


In [22]:
def run_custom_epochs(model, epochs=2, callback=None):
    optimizer_local = tf.keras.optimizers.Adam()
    accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
    if callback is not None:
        callback.set_model(model)
        callback.on_train_begin()
    for epoch in itertools.islice(range(epochs), 2):
        accuracy.reset_state()
        mean_loss = tf.keras.metrics.Mean()
        for batch_x, batch_y in loop_data:
            with tf.GradientTape() as tape:
                logits = model(batch_x, training=True)
                loss = loop_loss(batch_y, logits)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer_local.apply_gradients(zip(gradients, model.trainable_variables))
            accuracy.update_state(batch_y, logits)
            mean_loss.update_state(loss)
        logs = {'loss': float(mean_loss.result()), 'accuracy': float(accuracy.result())}
        print(epoch + 1, logs)
        if callback is not None:
            callback.on_epoch_end(epoch, logs)
    if callback is not None:
        callback.on_train_end()
accuracy_model = make_loop_model()
run_custom_epochs(accuracy_model)

1 {'loss': 1.9936890602111816, 'accuracy': 0.3671875}
2 {'loss': 1.2776585817337036, 'accuracy': 0.755859375}


<details>
<summary>Click here for solution</summary><br>

```python
# Import necessary libraries
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data() 
x_train = x_train / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy() 

# Step 4: Implement the Custom Training Loop with Accuracy Tracking
epochs = 5 
for epoch in range(epochs): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()} Accuracy = {accuracy_metric.result().numpy()}') 
    accuracy_metric.reset_state() 


### Exercise 3: Custom Callback for Advanced Logging 

#### Objective: Implement a custom callback to log additional metrics and information during training. 

#### Instructions: 

1. Set up the environment and define the model, loss function, optimizer, and metric. 

2. Create a custom callback to log additional metrics at the end of each epoch. 

3. Implement the custom training loop with the custom callback. 


In [23]:
class AdvancedLogging(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):
        print({'epoch': epoch + 1, 'learning_rate': float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)) if getattr(self.model, 'optimizer', None) else 'custom-loop optimizer', 'metrics': dict(logs or {})})
logged_model = make_loop_model()
run_custom_epochs(logged_model, callback=AdvancedLogging())

1 {'loss': 1.925249695777893, 'accuracy': 0.447265625}
{'epoch': 1, 'learning_rate': 'custom-loop optimizer', 'metrics': {'loss': 1.925249695777893, 'accuracy': 0.447265625}}


2 {'loss': 1.2019062042236328, 'accuracy': 0.755859375}
{'epoch': 2, 'learning_rate': 'custom-loop optimizer', 'metrics': {'loss': 1.2019062042236328, 'accuracy': 0.755859375}}


<details>
<summary>Click here for solution</summary> </br>

```python
# Import necessary libraries
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.callbacks import Callback 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train = x_train / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    tf.keras.Input(shape=(28, 28)),  # Updated Input layer syntax
    Flatten(), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam() 
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy() 

# Step 4: Implement the Custom Callback
class CustomCallback(Callback): 
    def on_epoch_end(self, epoch, logs=None): 
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}') 

# Step 5: Implement the Custom Training Loop with Custom Callback
custom_callback = CustomCallback() 

for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    custom_callback.on_epoch_end(epoch, logs={'loss': loss.numpy(), 'accuracy': accuracy_metric.result().numpy()}) 
    accuracy_metric.reset_state()  # Updated method



### Exercise 5: Add Hidden layer and the Output Layer 

#### Objective: Add couple of hidden layer to help the model learn complex patterns in the data.
### Define the output layer of a neural network for a multi-class classification problem using the Keras Functional API. 

#### Instructions: 
- Define 2 hidden layers, Input layer(of shape 28,28) as the parameter for the first hidden layer)

- Using the `hidden_layer2` as the input, add a `Dense` output layer. 

- The output layer with `sigmoid function` as activation function


In [24]:
from tensorflow.keras.layers import Input, Dense, Flatten
input_layer = Input(shape=(28, 28))
flatten = Flatten()(input_layer)
hidden_layer1 = Dense(64, activation='relu')(flatten)
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)
output_layer = Dense(10, activation='softmax')(hidden_layer2)

<details>
<summary>Click here for solution</summary> </br>

```python
from tensorflow.keras.layers import Input, Dense, Flatten

# Re-define layers for MNIST (28x28 images, 10 digit classes)
input_layer   = Input(shape=(28, 28))                          # Input: 28x28 grayscale images
flatten       = Flatten()(input_layer)                         # Flatten to 784-dim vector
hidden_layer1 = Dense(64, activation='relu')(flatten)          # First hidden layer
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)    # Second hidden layer
output_layer  = Dense(10, activation='softmax')(hidden_layer2) # Output: 10 classes (digits 0-9)
 ```   

</details>


### Exercise 6: Create the Model 

#### Objective: Create a Keras Functional API model by connecting the input and output layers defined in the previous exercises. 

#### Instructions: 

- Use `tf.keras.Model` to create the model. 

- Pass `input_layer` as the `inputs` argument and `output_layer` as the `outputs` argument. 


In [25]:
model = Model(inputs=input_layer, outputs=output_layer)

<details>
<summary>Click here for solution</summary> </br>

```python
# Create the model by specifying input and output layers
model = Model(inputs=input_layer, outputs=output_layer)
 ```   

</details>


### Exercise 7: Compile the Model 

#### Objective: Configure the model for training by specifying the optimizer, loss function, and evaluation metric. 

#### Instructions: 

- Compile the model using the **Adam** optimizer. 

- Use **`sparse_categorical_crossentropy`** as the loss function
- Include **`accuracy`** as the evaluation metric. 


In [26]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

<details>
<summary>Click here for solution</summary> </br>

```python
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct loss for integer labels (0-9)
    metrics=['accuracy']
)
```

</details>


### Exercise 8: Train the Model 

#### Objective: Train the compiled model on the MNIST training dataset using `model.fit()`. 

#### Instructions: 

- Call `model.fit()` with the training data `x_train` and labels `y_train`. 

- Train for **5 epochs** with a **batch size of 32**. 


In [27]:
history = model.fit(x_train, y_train, epochs=2, batch_size=32)
print('Training complete.')

Epoch 1/2


 1/16 ━━━━━━━━━━━━━━━━━━━━ 3s 254ms/step - accuracy: 0.1250 - loss: 2.3082

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1908 - loss: 2.2371  


Epoch 2/2


 1/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5312 - loss: 1.8293

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 979us/step - accuracy: 0.6111 - loss: 1.6809


Training complete.


<details>
<summary>Click here for solution</summary> </br>

```python
history = model.fit(
    x_train, y_train,        # Training data and labels
    epochs=5,                # Number of full passes over the training data
    batch_size=32,           # Number of samples per gradient update
)

print('Training complete.')
```

</details>


### Exercise 9: Evaluate the Model 

#### Objective: Assess the trained model's performance on unseen test data using `model.evaluate()`. 

#### Instructions: 

- Call `model.evaluate()` with the test data `x_test` and labels `y_test`. 

- Capture the returned **test loss** and **test accuracy**. 

- Print both values to summarise the model's generalisation performance. 


In [28]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6875 - loss: 1.3365

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6745 - loss: 1.3559 


Test loss:     1.3817
Test accuracy: 0.6719


<details>
<summary>Click here for solution</summary> </br>

```python
# Evaluate the trained model on the held-out test dataset
test_loss, test_accuracy = model.evaluate(x_test, y_test)

# Print the evaluation results
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
```

</details>


### Conclusion: 

Congratulations on completing this lab! You have now successfully created, trained, and evaluated a simple neural network model using the Keras Functional API. This foundational knowledge will allow you to build more complex models and explore advanced functionalities in Keras. 


Copyright © IBM Corporation. All rights reserved.
